---
## CSCI 3202 Homework #3
### Due Monday, Februay 9 by 11:59 pm
### 100 Points

<br>

### Your name: Jeffrey Allen

<br>

---

In [47]:
from probability4e import *
from utils4e import print_table

This homework uses the aima python github repository. You will need a python function from the aima-python folder. Since we will be using the function in this repository quite a bit, it is probably easiest to clone it:

git clone https://github.com/aimacode/aima-python.git

Once you have the repository, you will need to copy the .py file(s) you need into the directory containing your homework, or copy the the homework notebook (not the folder containing the .ipynb file) into the aima-data directory.

For this assignment, you will need the file:

probability4e.py\
utils4e.py

Note: You should copy the file(s) from the repository, not move them. If you move the files, they won't be in the repository for your next assignment.

After you do this, if the import statement below fails, you will need to restart either the Jupyter server or your Python kernel. We are restarting to clear the cache for the import statement.

Use enumeration_ask() from the AIMA repository to solve the problems below


---
### Problem 1: Football Play Prediction Network

In this problem, we will analyze a Bayesian network that models offensive plays in American football. The network helps predict play outcomes based on various game conditions and decisions.

### Network Structure:

The Bayesian network models the following scenario:

**Parent Nodes (no dependencies):**
- **StarPlayerOnField**: Whether the team's star offensive player is on the field
- **DefenseBlitz**: Whether the defense is blitzing (sending extra pass rushers)

**Intermediate Node:**
- **PassPlay**: Whether the offense calls a pass play (vs. run play)
  - Depends on: StarPlayerOnField, DefenseBlitz

**Outcome Nodes:**
- **Touchdown**: Whether the play results in a touchdown
  - Depends on: PassPlay
- **FirstDown**: Whether the play results in a first down (but not a touchdown)
  - Depends on: PassPlay

**Network Diagram:**
```
    StarPlayerOnField     DefenseBlitz
              \              /
               \            /
                \          /
                 PassPlay
                 /      \
                /        \
               /          \
         Touchdown      FirstDown
```

The above problem is a Bayesian network for football play prediction. We have two parent nodes with no dependencies: _StarPlayerOnField_ and _DefenseBlitz_. We have one node with two parents: _PassPlay_, which leads to two child nodes representing outcomes: _Touchdown_ and _FirstDown_.

You may refer to the [probability module](https://github.com/aimacode/aima-python/blob/master/probability4e.py) and the [probability notebook](https://github.com/aimacode/aima-python/blob/master/probability4e.ipynb) for more explanations.

In [48]:
T, F = True, False

football = BayesNet([
    ('StarPlayerOnField', '', 0.75),
    ('DefenseBlitz', '', 0.35),
    ('PassPlay', ['StarPlayerOnField', 'DefenseBlitz'],
     {(T, T): 0.42, (T, F): 0.68, (F, T): 0.28, (F, F): 0.55}),
    ('Touchdown', 'PassPlay', {T: 0.15, F: 0.08}),
    ('FirstDown', 'PassPlay', {T: 0.45, F: 0.52})
])
print(football.variables) # prints out all nodes present in the BayesNet

['StarPlayerOnField', 'DefenseBlitz', 'PassPlay', 'Touchdown', 'FirstDown']


### Questions (40 points):

---
### Solve by hand

1.1 What is the probability of a pass play being called? Write the equations needed. Your equations need to include only the probabilities and sums necessary to solve the problem. Substitute the numbers from the conditional probability tables above and solve for a numerical value of the probability.

**Equations:**

Write your equations here before computing the answer (5 points for equations, 5 points for answer).

# Write your solution here
$ P(PassPlay) = \sum_{s} \sum_{b} P(PassPlay | s,b)P(s)P(b) $ \
$ P(Passplay) = (0.42)(0.75)(0.35)+(0.68)(0.75)(0.65)+(0.28)(0.25)(0.35)+(0.55)(0.25)(0.65) = 0.555625$

1.2 What is the probability of scoring a touchdown? Write the solution symbolically first. Your equation needs to include only the probabilities and sums necessary to solve the problem. Substitute the numbers from the conditional probability tables above and solve for a numerical value of the probability. You may use results from earlier questions if they are helpful.

**Equations:**

Write your equations here before computing the answer (5 points for equations, 5 points for answer).

# Write your solution here
$P(Touchdown)=P(Touchdown∣PassPlay)P(PassPlay)+P(Touchdown∣\neg PassPlay)P(\neg PassPlay)$ \
$P(Touchdown) = (0.15)(0.555625)+(0.08)(1−0.555625) = 0.11889$

1.3 What is the probability that the star player is on the field given that a pass play was called? Write the solution symbolically first. Your equation needs to include only the probabilities and sums necessary to solve the problem. Substitute the numbers from the conditional probability tables above and solve for a numerical value of the probability.

$P(StarPlayerOnField∣PassPlay)=\frac{P(PassPlay∣StarPlayerOnField)P(StarPlayerOnField)}{P(PassPlay)}$ \
$P(PassPlay∣StarPlayerOnField) = \sum_{b} P(PassPlay∣StarPlayerOnField,b)P(b)=(0.42)(0.35)+(0.68)(0.65)=0.589$

**Equations:**

Write your equations here before computing the answer (5 points for equations, 5 points for answer).

# Write your solution here
$P(StarPlayerOnField∣PassPlay)=\frac{P(PassPlay∣StarPlayerOnField)P(StarPlayerOnField)}{P(PassPlay)}$ \
$P(PassPlay∣StarPlayerOnField) = \sum_{b} P(PassPlay∣StarPlayerOnField,b)P(b)=(0.42)(0.35)+(0.68)(0.65)=0.589$ \ $P(StarPlayerOnField∣PassPlay) = \frac{(0.589)(0.75)}{0.55625} = 0.795$

1.4 Using enumeration, write the equation for P(FirstDown). Your equation must include the joint distribution for all 5 of the variables and the sums necessary to solve for P(FirstDown). **You only need to write the equation. You do not need to solve for a numerical answer**.

**Solution:**

Either type your solution using LaTeX or take a photo and attach your solution here (10 points for equations).

$\sum_{S} \sum_{B} \sum_{P} \sum_{T} P(S)P(B)P(P|S,B)P(T|P)P(FirstDown|P) $

---
### Solve through code (20 points)

In [49]:
#just for me to reference when using
def elimination_ask(X, e, bn):
    """
    Compute bn's P(X|e) by variable elimination. [Figure 13.12]
    >>> elimination_ask('Burglary', dict(JohnCalls=T, MaryCalls=T), burglary
    ...  ).show_approx()
    'False: 0.716, True: 0.284'
    """
    assert X not in e, "Query variable must be distinct from evidence"
    factors = []
    for var in reversed(bn.variables):
        factors.append(make_factor(var, e, bn))
        if is_hidden(var, X, e):
            factors = sum_out(var, factors, bn)
    return pointwise_product(factors, bn).normalize()

2.1 What is the probability of getting a first down (but not a touchdown) (5 points)?

In [50]:
# Write your solution here
p_fd_not_td = enumerate_all(
    football.variables,
    {'FirstDown': T, 'Touchdown': F},
    football
)

print(p_fd_not_td)

0.4251155625


2.2 The defense is blitzing. What is the probability that the offense will call a pass play (5 points)?

In [51]:
# Write your solution here
p_pass_given_blitz = enumeration_ask(
    'PassPlay',
    {'DefenseBlitz': T},
    football
)

print(p_pass_given_blitz.show_approx())

False: 0.615, True: 0.385


2.3 The star player is on the field and the defense is not blitzing. What is the probability of scoring a touchdown (5 points)?

In [52]:
# Write your solution here
p_td_given_conditions = enumeration_ask(
    'Touchdown',
    {'StarPlayerOnField': T, 'DefenseBlitz': F},
    football
)

print(p_td_given_conditions.show_approx())

False: 0.872, True: 0.128


2.4 The offense scored a touchdown. What is the probability that it was a pass play (5 points)?

In [53]:
# Write your solution here
p_pass_given_td = enumeration_ask(
    'PassPlay',
    {'Touchdown': T},
    football
)

print(p_pass_given_td.show_approx())

False: 0.299, True: 0.701


---
## Problem 3: Advanced Football Strategy Network

This problem explores a more complex Bayesian network for football game strategy and player performance.

### Network Structure:

This network models quarterback performance and offensive success:

**Independent Parent Nodes:**
- **WeatherConditions**: Whether weather conditions are good (False = bad weather, True = good weather)
- **OpposingDefenseRank**: Whether the opposing defense is highly ranked (strong)

**Dependent Intermediate Nodes:**
- **QBPressure**: Whether the quarterback faces heavy pressure
  - Depends on: OpposingDefenseRank
- **PassAccuracy**: Whether the quarterback has high pass accuracy
  - Depends on: WeatherConditions, QBPressure
- **ReceiverOpen**: Whether receivers are getting open
  - Depends on: OpposingDefenseRank

**Dependent Outcome Nodes:**
- **CompletedPass**: Whether the pass is completed
  - Depends on: PassAccuracy, ReceiverOpen
- **BigPlay**: Whether the play results in a gain of 20+ yards
  - Depends on: CompletedPass
- **Turnover**: Whether the play results in a turnover (interception/fumble)
  - Depends on: QBPressure, PassAccuracy

**Network Diagram:**
```
WeatherConditions          OpposingDefenseRank
        |                    /              \
        |                   /                \
        |             QBPressure          ReceiverOpen
        |              /    |                  |
        |             /     |                  |
        |            /      |                  |
         \          /       |                  |
          \        /        |                  |
         PassAccuracy       |                  |
              |    \        |                  |
              |     \       |                 /
              |      \      |                /
              |       \     |               /
              |        Turnover            /
              |                           /
               \                         /
                \                       /
                 \                     /
                  CompletedPass       /
                        |            /
                        |           /
                      BigPlay
```

In [54]:
T, F = True, False

advanced_football = BayesNet([
    ('WeatherConditions', '', 0.65),  # P(good weather)
    ('OpposingDefenseRank', '', 0.40),  # P(strong defense)
    ('QBPressure', 'OpposingDefenseRank',
     {T: 0.70, F: 0.30}),  # P(pressure | defense rank)
    ('PassAccuracy', ['WeatherConditions', 'QBPressure'],
     {(T, T): 0.55, (T, F): 0.82, (F, T): 0.38, (F, F): 0.68}),
    ('ReceiverOpen', 'OpposingDefenseRank',
     {T: 0.35, F: 0.72}),  # P(receiver open | defense rank)
    ('CompletedPass', ['PassAccuracy', 'ReceiverOpen'],
     {(T, T): 0.88, (T, F): 0.62, (F, T): 0.58, (F, F): 0.28}),
    ('BigPlay', 'CompletedPass',
     {T: 0.22, F: 0.03}),  # P(big play | completion status)
    ('Turnover', ['QBPressure', 'PassAccuracy'],
     {(T, T): 0.08, (T, F): 0.25, (F, T): 0.03, (F, F): 0.12})
])
print(advanced_football.variables) # prints out all nodes present in the BayesNet

['WeatherConditions', 'OpposingDefenseRank', 'QBPressure', 'PassAccuracy', 'ReceiverOpen', 'CompletedPass', 'BigPlay', 'Turnover']


### Questions (40 points)

3.1 What is the overall probability of completing a pass (5 points)?

In [55]:
# Write your solution here
p_completed_pass = enumeration_ask(
    'CompletedPass',
    {},
    advanced_football
)

print(p_completed_pass.show_approx())

False: 0.345, True: 0.655


3.2 The team is facing a strong defense in bad weather conditions. What is the probability of completing a pass (5 points)?

In [56]:
# Write your solution here
p_completed_given_conditions = enumeration_ask(
    'CompletedPass',
    {'OpposingDefenseRank': T, 'WeatherConditions': F},
    advanced_football
)

print(p_completed_given_conditions.show_approx())

False: 0.462, True: 0.538


3.3 The team completed a pass for a big play. What is the probability that they were facing a weak defense (OpposingDefenseRank = False) (5 points)?

In [57]:
# Write your answer here
p_weak_defense_given_big_play = enumeration_ask(
    'OpposingDefenseRank',
    {'BigPlay': T},
    advanced_football
)

print(p_weak_defense_given_big_play.show_approx())

False: 0.641, True: 0.359


3.4 In your own words, explain how this Bayesian network demonstrates both causal and diagnostic reasoning in the context of football analytics. Provide at least one example of each type of reasoning from this network (10 points).

**Your answer here:**



This Bayesian networks shows causal reasoning by modeling how underlying game conditions lead to observable outcomes. Like how in poor weather and high quarter back pressure results in reduced pass accuracy which directly lowers the probability of a competed pass.

This network also shows diagnostic reasoning by allowing inference in the reverse direction where outcomes can give information on likely causes. For example, when a completed pass that resulted in a big play, has a high probability that the opposing defense was weak.

3.5 How could a football coach use this Bayesian network to make strategic decisions during a game? Consider comparing the probability of success under different conditions (e.g., good weather vs. bad weather, strong defense vs. weak defense). Provide specific probability calculations to support your analysis (15 points, 10 points for reasoned answer, 5 points for support calculations).

In [58]:
# Calculate P(CompletedPass | good weather, weak defense)
p_cp_good_weather_weak_def = enumeration_ask(
    'CompletedPass',
    {'WeatherConditions': T, 'OpposingDefenseRank': F},
    advanced_football
)

print(p_cp_good_weather_weak_def.show_approx())

#This calculation shows how likely a pass is to be completed when conditions favor the offense. 
#Good weather improves pass accuracy, and a weak defense reduces pressure and increases receiver separation. A high probability here supports calling more passing plays.

False: 0.274, True: 0.726


In [59]:
# Calculate P(CompletedPass | bad weather, strong defense)
p_cp_bad_weather_strong_def = enumeration_ask(
    'CompletedPass',
    {'WeatherConditions': F, 'OpposingDefenseRank': T},
    advanced_football
)

print(p_cp_bad_weather_strong_def.show_approx())
#This calculation measures pass success under unfavorable conditions. Bad weather lowers accuracy, and a strong defense increases pressure and coverage quality. 
#A lower probability suggests passing is riskier and less effective in this scenario.

False: 0.462, True: 0.538


In [60]:
# Calculate P(Turnover | good weather, weak defense)
p_to_good_weather_weak_def = enumeration_ask(
    'Turnover',
    {'WeatherConditions': T, 'OpposingDefenseRank': F},
    advanced_football
)

print(p_to_good_weather_weak_def.show_approx())
#This calculation estimates the risk of a turnover when conditions are favorable. With good weather and less defensive pressure, 
# the quarterback is less likely to make mistakes. A low turnover probability supports aggressive offensive play-calling.

False: 0.921, True: 0.0793


In [61]:
# Calculate P(Turnover | bad weather, strong defense)
p_to_bad_weather_strong_def = enumeration_ask(
    'Turnover',
    {'WeatherConditions': F, 'OpposingDefenseRank': T},
    advanced_football
)

print(p_to_bad_weather_strong_def.show_approx())
#This calculation evaluates turnover risk under difficult conditions. Strong defensive pressure and poor weather both increase the chance of mistakes. 
# A higher turnover probability suggests the offense should play conservatively to avoid losing possession.

False: 0.853, True: 0.147
